# GPU memory growth (put near top)

In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
print("GPUs:", gpus)

GPUs: []


# Cell 1 — Load memmaps + trained model

In [3]:
import numpy as np, json
from pathlib import Path
import tensorflow as tf

UNLABELED_DIR = Path(r"C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\features_mfcc_unlabeled")
LABELED_DIR   = Path(r"C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\features_mfcc_labeled")

X_unl = np.load(UNLABELED_DIR / "X_unlabeled.npy", mmap_mode="r")
with open(UNLABELED_DIR / "unlabeled_index.json", "r", encoding="utf-8") as f:
    index = json.load(f)

model = tf.keras.models.load_model(LABELED_DIR / "cnn_mfcc_best.keras")

print("X_unl:", X_unl.shape, X_unl.dtype)
print("Index entries (files):", len(index))

X_unl: (687350, 3, 32, 201) float16
Index entries (files): 11650


# Cell 2 — Build a prediction dataset for unlabeled (memmap-safe)

This streams batches from ```X_unl``` and never loads it all.

In [4]:
THRESH = 0.95
BATCH_PRED = 64  # drop to 32 if memory warnings persist

file_preds = []  # one row per file: file, y, conf, start, nseg

for i, item in enumerate(index, 1):
    start = item["start"]
    nseg  = item["n_segments"]
    file_path = item["file"]

    # Accumulate probabilities in chunks
    prob_sum = np.zeros((4,), dtype=np.float64)
    seen = 0

    for j in range(0, nseg, BATCH_PRED):
        xb = np.array(X_unl[start+j : start+min(nseg, j+BATCH_PRED)], dtype=np.float32)
        pb = model.predict(xb, verbose=0)  # (batch, 4)
        prob_sum += pb.sum(axis=0)
        seen += pb.shape[0]

    prob_mean = (prob_sum / max(1, seen)).astype(np.float32)
    y_hat = int(prob_mean.argmax())
    conf  = float(prob_mean.max())

    file_preds.append({
        "file": file_path,
        "start": int(start),
        "n_segments": int(nseg),
        "pseudo_y": y_hat,
        "conf": conf
    })

    if i % 200 == 0 or i == len(index):
        accepted = sum(1 for r in file_preds if r["conf"] >= THRESH)
        print(f"Processed {i}/{len(index)} files | accepted so far: {accepted}")

Processed 200/11650 files | accepted so far: 92
Processed 400/11650 files | accepted so far: 187
Processed 600/11650 files | accepted so far: 244
Processed 800/11650 files | accepted so far: 371
Processed 1000/11650 files | accepted so far: 525
Processed 1200/11650 files | accepted so far: 684
Processed 1400/11650 files | accepted so far: 766
Processed 1600/11650 files | accepted so far: 804
Processed 1800/11650 files | accepted so far: 848
Processed 2000/11650 files | accepted so far: 906
Processed 2200/11650 files | accepted so far: 992


KeyboardInterrupt: 

# Cell 3 — Generate pseudo-labels with confidence filtering

This creates small arrays of ```selected_indices```, ```pseudo_labels```, and ```pseudo_confidence```.

In [ ]:
accepted = [r for r in file_preds if r["conf"] >= THRESH]
print("Accepted files:", len(accepted), "of", len(file_preds))

# Optional: cap accepted files per class to avoid collapse into class 3
MAX_FILES_PER_CLASS = 2000
accepted_sorted = sorted(accepted, key=lambda r: -r["conf"])  # highest confidence first

per_class = {0:0,1:0,2:0,3:0}
accepted_capped = []
for r in accepted_sorted:
    k = r["pseudo_y"]
    if per_class[k] < MAX_FILES_PER_CLASS:
        accepted_capped.append(r)
        per_class[k] += 1

print("Accepted capped per class:", per_class)
accepted = accepted_capped

# Expand: every segment in accepted files becomes a training sample
pseudo_seg_idx = []
pseudo_seg_y   = []

for r in accepted:
    s = r["start"]
    n = r["n_segments"]
    pseudo_seg_idx.extend(range(s, s+n))
    pseudo_seg_y.extend([r["pseudo_y"]] * n)

pseudo_seg_idx = np.array(pseudo_seg_idx, dtype=np.int64)
pseudo_seg_y   = np.array(pseudo_seg_y, dtype=np.int64)

print("Pseudo-labeled segments:", len(pseudo_seg_idx))
u, c = np.unique(pseudo_seg_y, return_counts=True)
print("Pseudo segment class counts:", dict(zip(u, c)))

prevent pseudo labels from being 95% class “3”. If your selected set is overwhelmingly class 3, cap per class to keep diversity:

In [ ]:
BATCH_TRAIN = 64
SEED = 123

X_unl_global = X_unl
idx_global = pseudo_seg_idx
y_global = pseudo_seg_y

def _get_pseudo(i):
    i = int(i)
    real = int(idx_global[i])
    x = np.array(X_unl_global[real], dtype=np.float32)  # one sample from disk
    y = np.int64(y_global[i])
    return x, y

def tf_get_pseudo(i):
    x, y = tf.py_function(_get_pseudo, [i], [tf.float32, tf.int64])
    x.set_shape((3, 32, 201))
    y.set_shape(())
    return x, y

pseudo_ds = tf.data.Dataset.from_tensor_slices(np.arange(len(idx_global), dtype=np.int64))
pseudo_ds = pseudo_ds.shuffle(min(len(idx_global), 20000), seed=SEED, reshuffle_each_iteration=True)
pseudo_ds = pseudo_ds.map(tf_get_pseudo, num_parallel_calls=tf.data.AUTOTUNE)
pseudo_ds = pseudo_ds.batch(BATCH_TRAIN).prefetch(tf.data.AUTOTUNE)

print("Pseudo ds ready.")

# Cell 6 — Evaluate on test

In [ ]:
# Labeled loaders (memmap-safe)
X_train = np.load(LABELED_DIR / "X_train.npy", mmap_mode="r")
y_train = np.load(LABELED_DIR / "y_train.npy")
X_val   = np.load(LABELED_DIR / "X_val.npy",   mmap_mode="r")
y_val   = np.load(LABELED_DIR / "y_val.npy")

def make_labeled_ds(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(20000, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_TRAIN)
    ds = ds.map(lambda a,b: (tf.cast(a, tf.float32), tf.cast(b, tf.int64)),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds_labeled = make_labeled_ds(X_train, y_train, training=True)
val_ds = make_labeled_ds(X_val, y_val, training=False)

# Mix: start conservative
train_ds_mixed = tf.data.Dataset.sample_from_datasets(
    [train_ds_labeled, pseudo_ds],
    weights=[0.7, 0.3],
    seed=SEED
)

# Fine-tune with lower LR
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", mode="max", patience=3, factor=0.5),
]

history = model.fit(train_ds_mixed, validation_data=val_ds, epochs=30, callbacks=callbacks)